In [ ]:
import os
# os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"

In [ ]:
import torch
import evaluate
import numpy as np
from peft import UIOrthoLoRAConfig, UILinLoRAConfig, get_peft_model, TaskType, PeftConfig, PeftModel
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    TrainingArguments, Trainer, DataCollatorForLanguageModeling
)
from datasets import load_dataset

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# Load GPT-2 tokenizer with use_fast=False for full control
tok = AutoTokenizer.from_pretrained("gpt2", use_fast=False)

# BEFORE: GPT-2 has no pad token
print("Before:", tok.pad_token, tok.pad_token_id)

# Add a real PAD token (ID will be 50257)
tok.add_special_tokens({"pad_token": "[PAD]"})

print("After:", tok.pad_token, tok.pad_token_id)
print("PAD token ID:", tok.convert_tokens_to_ids("[PAD]"))

# Create a short batch of prompt_ids
examples = [
    tok.convert_tokens_to_ids(tok.tokenize("name[Bibimbap House] =>")),
    tok.convert_tokens_to_ids(tok.tokenize("name[Wildwood] => food[Italian]"))
]

# Left pad manually
pad_id = tok.pad_token_id
max_len = max(len(x) for x in examples)

for i, seq in enumerate(examples):
    pad = [pad_id] * (max_len - len(seq))
    padded = pad + seq
    print(f"\nExample {i+1}")
    print("Raw IDs:     ", padded)
    print("Tokens:      ", tok.convert_ids_to_tokens(padded))
    print("Last token:  ", padded[-1], tok.convert_ids_to_tokens([padded[-1]])[0])
    print("Padding OK?  ", padded[-1] != pad_id)


In [ ]:
from datasets import load_dataset

ds = load_dataset("tuetschek/e2e_nlg")

# View a few examples
for i in range(10):
    record = ds["test"][i]
    print("Meaning Representation (MR):", record["meaning_representation"])
    print("Reference:", record.get("human_reference") or record.get("reference"))
    print("-" * 60)


In [ ]:
len(ds['test'])

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer

# Load test set
ds = load_dataset("tuetschek/e2e_nlg")
test_ds = ds["test"]

# Load tokenizer (use the same as your model)
tokenizer = AutoTokenizer.from_pretrained("gpt2")  # or your model's tokenizer

# Detokenize model predictions
def decode_preds(preds):
    return tokenizer.batch_decode(preds, skip_special_tokens=True)

# Print model generations vs gold labels
def print_preds_vs_gold(model_preds, labels):
    for i in range(min(10, len(model_preds))):
        pred = model_preds[i].strip()
        gold = labels[i].strip()
        print(f"💡 Prediction {i+1}: {pred}")
        print(f"✅ Gold Label {i+1}: {gold}")
        print("-" * 80)


In [ ]:
ds

In [ ]:
for i, sentence in enumerate(ds["test"]["human_reference"]):
    print(f"Sentence {i+1}: {sentence}")
    print("-" * 80)

    if i == 20:
        break

In [ ]:
from torch.utils.data import DataLoader
INFERENCE_ARGS = {
    "num_beams": 10,
    "no_repeat_ngram_size": 4,
    "length_penalty": 0.9,
    "max_new_tokens": 64,
}
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
def set_tokenizer(tokenizer):
    tokenizer.padding_side = "left"
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

def set_contiguous(model):
    for m in model.modules():
        if hasattr(m, "parametrizations") and "weight" in m.parametrizations:
            base = m.parametrizations.weight[0].base
            if not base.is_contiguous():
                base.data = base.data.contiguous()

def get_tokenizer_and_model(model_path: str, device):
    """
    Load a base model and inject a saved PEFT adapter from `model_path`.
    """
    # 1) Load the adapter config to get the original base model
    peft_config = PeftConfig.from_pretrained(model_path)
    base_model_name = peft_config.base_model_name_or_path

    # 2) Load base model and tokenizer
    tokenizer = AutoTokenizer.from_pretrained(base_model_name, use_fast=False)
    set_tokenizer(tokenizer)

    base_model = AutoModelForCausalLM.from_pretrained(base_model_name)
    base_model.config.pad_token_id = tokenizer.pad_token_id
    base_model.generation_config.pad_token_id = tokenizer.pad_token_id
    base_model = base_model.to(device)

    # 3) Load the adapter into the base model
    model = PeftModel.from_pretrained(base_model, model_path)
    model = model.to(device)

    # 4) Ensure contiguous weights (optional)
    set_contiguous(model)

    return tokenizer, model, peft_config

In [ ]:
tokenizer, model, peft_config = get_tokenizer_and_model("outputs/models/lr_5e-3", device)

In [ ]:
def collate_fn(batch):
        feats = [{"input_ids": b["prompt_ids"]} for b in batch]
        out = tokenizer.pad(
        feats,
        padding="longest",
        return_attention_mask=True,
        return_tensors="pt"
    )
        return out        

dataloader = DataLoader(
    ds["test"],
    batch_size=8,
    collate_fn=collate_fn,
)

for item in tqdm(dataloader):
    prompt_ids = item["input_ids"].to(device)
    attention_mask = item["attention_mask"].to(device)

    with torch.no_grad():
        outputs = model.generate(
            input_ids=prompt_ids,
            attention_mask=attention_mask,
            max_new_tokens=INFERENCE_ARGS["max_new_tokens"],
            num_beams=INFERENCE_ARGS["num_beams"],
            no_repeat_ngram_size=INFERENCE_ARGS["no_repeat_ngram_size"],
            length_penalty=INFERENCE_ARGS["length_penalty"],
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    preds = tokenizer.batch_decode(outputs, skip_special_tokens=True)
    # strip the prompt text (“=>” part) to isolate hypothesis
    preds = [p.split("=>")[-1].strip() for p in preds]
    gen_texts.extend(preds)

In [ ]:
def load_and_prepare(tokenizer, max_length=128):
    """Load E2E dataset and prepare tokenised fields."""
    ds = load_dataset("tuetschek/e2e_nlg")

    def linearise(record):
        mr = record["meaning_representation"]  # e.g. "name[Bibimbap House], food[Indian]"
        ref = record["human_reference"] if "human_reference" in record else record["reference"]
        prompt = f"{mr} => "  # simple prompt pattern
        example = prompt + ref
        tokenised = tokenizer(
            example,
            truncation=True,
            max_length=max_length,
            padding="max_length",
        )
        labels = tokenised["input_ids"].copy()
        # Mask prompt tokens so they are ignored in the loss (label = -100)
        prompt_ids = tokenizer(prompt, add_special_tokens=False)["input_ids"]
        prompt_len = len(prompt_ids)
        labels[:prompt_len] = [-100] * prompt_len
        tokenised["labels"] = labels
        return tokenised

    ds = ds.map(linearise, remove_columns=ds["train"].column_names)
    return ds

In [ ]:
from pycocoevalcap.cider.cider import Cider

class CiderMetric:
    """Wraps pycocoevalcap so it looks like an `evaluate` metric."""
    def __init__(self):
        self.scorer = Cider()

    def compute(self, *, predictions, references):
        # pycocoevalcap expects dicts: {idx: ["sentence"]}
        hyps = {i: [pred] for i, pred in enumerate(predictions)}
        refs = {i: [ref]  for i, ref  in enumerate(references)}
        score, _ = self.scorer.compute_score(refs, hyps)
        return {"cider": score}

cider_metric = CiderMetric()

In [ ]:
bleu_metric = evaluate.load("sacrebleu")
meteor_metric = evaluate.load("meteor")
rouge_metric = evaluate.load("rouge")
nist_metric = evaluate.load("nist_mt")

In [ ]:
def postprocess_text(preds, labels):
    preds = [p.strip() for p in preds]
    labels = [l.strip() for l in labels]
    return preds, labels


def set_contiguous(model):
    for m in model.modules():
        if hasattr(m, "parametrizations") and "weight" in m.parametrizations:
            base = m.parametrizations.weight[0].base
            if not base.is_contiguous():
                base.data = base.data.contiguous()


# ------------------------------------------------------------------
# helper -----------------------------------------------------------
def _postprocess_strs(predictions, references):
    """Strip leading/trailing spaces & unify whitespace."""
    preds = [p.strip() for p in predictions]
    refs  = [r.strip() for r in references]
    return preds, refs
# ------------------------------------------------------------------



def compute_metrics(eval_pred):
    """Compute BLEU, METEOR, ROUGE-L, (optionally) NIST on E2E-NLG."""
    preds, labels = eval_pred

    # ── tensors → numpy ───────────────────────────────────────────────
    if isinstance(preds, torch.Tensor):
        preds = preds.cpu().numpy()
    if isinstance(labels, torch.Tensor):
        labels = labels.cpu().numpy()

    # ── logits → ids if necessary ─────────────────────────────────────
    if preds.ndim == 3:                      # (batch, seq, vocab)
        preds = preds.argmax(-1)

    # ── un-mask labels ────────────────────────────────────────────────
    labels = labels.copy()
    labels[labels == -100] = tokenizer.pad_token_id

    # ── decode ────────────────────────────────────────────────────────
    pred_strs  = tokenizer.batch_decode(preds,  skip_special_tokens=True)
    label_strs = tokenizer.batch_decode(labels, skip_special_tokens=True)
    pred_strs  = [s.strip() for s in pred_strs]
    label_strs = [s.strip() for s in label_strs]

    # ── metrics ───────────────────────────────────────────────────────
    bleu   = bleu_metric.compute(
                predictions=pred_strs,
                references=[[r] for r in label_strs]
             )["score"]

    meteor = meteor_metric.compute(
                predictions=pred_strs,
                references=label_strs
             )["meteor"]

    rougeL = rouge_metric.compute(
                predictions=pred_strs,
                references=label_strs,
                use_stemmer=True
             )["rougeL"]

    cider = cider_metric.compute(
                predictions=pred_strs,
                references=label_strs   # wrapper handles dict-conversion
            )["cider"]

    # ---- NIST (may not exist on tiny samples) ------------------------
    nist_raw = nist_metric.compute(
                predictions=pred_strs,
                references=[[r] for r in label_strs]
              )
    nist_val = nist_raw.get("nist", nist_raw.get("score"))  # could be None

    # helper to round only numerics
    def _r(x):
        return round(float(x), 4) if isinstance(x, Number) else x

    out = {
        "bleu"  : _r(bleu),
        "meteor": _r(meteor),
        "rougeL": _r(rougeL),
        "cider"  : _r(cider),
    }
    if nist_val is not None:
        out["nist"] = _r(nist_val)

    return out


In [ ]:
orthoLoRAConfig = UIOrthoLoRAConfig(
    target_modules=["attn.c_attn", "attn.c_proj"],
    fan_in_fan_out         = True,   # GPT-2 matrices are (out, in)
    initial_scaler         = 0.1,    # scale of the diagonal Σ at init
    initial_sigma          = 0.1,    # std-dev for the trainable Σ entries
    uiortholora_alpha      = 1,
    uiortholora_dropout    = 0,
    num_svalues_to_adapt   = 2,       # adapt the top-4 singular values
    num_svectors_to_adapt  = 2,       # adapt the corresponding vectors
    task_type              = TaskType.CAUSAL_LM
)

In [ ]:
from peft import LoraConfig

lora_config = LoraConfig(
    target_modules=["attn.c_attn", "attn.c_proj"],
    r=2,                         # very low rank for easy debugging
    lora_alpha=1,               # no extra scaling
    lora_dropout=0.0,           # no dropout for deterministic behavior
    bias="none",                # keep bias untouched
    fan_in_fan_out=False,       # match GPT-2 shape: (out, in)
    task_type=TaskType.CAUSAL_LM
)


In [ ]:
model_path = "gpt2-medium"
seed=42

torch.manual_seed(seed)
np.random.seed(seed)


tokenizer = AutoTokenizer.from_pretrained(model_path)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

ds = load_and_prepare(tokenizer)

base_model = AutoModelForCausalLM.from_pretrained(model_path)
base_model.config.pad_token_id = tokenizer.pad_token_id

In [ ]:
peft_config = PeftConfig.from_pretrained("outputs/models")

In [ ]:
base_model_check = AutoModelForCausalLM.from_pretrained(peft_config.base_model_name_or_path)
base_model_check = base_model_check.to(device)

In [ ]:
model = PeftModel.from_pretrained(base_model_check, "outputs/models")

In [ ]:
stop

In [ ]:
base_model = base_model.to(device)


In [ ]:
base_model.device

In [ ]:
stop

In [ ]:
model = get_peft_model(base_model, orthoLoRAConfig)

In [ ]:
set_contiguous(model)
model.print_trainable_parameters()

In [ ]:
data_collator = DataCollatorForLanguageModeling(tokenizer, mlm=False)

training_args = TrainingArguments(
    output_dir="outputs/check",
    overwrite_output_dir=True,
    eval_strategy="no",
    save_strategy="no",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=8,
    eval_accumulation_steps=2,
    learning_rate=1e-3,
    num_train_epochs=5,
    weight_decay=0.01,
    logging_steps=50,
    save_total_limit=1,
    report_to="none",
)

In [ ]:
trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=ds["train"].select(range(500)),
        eval_dataset=ds["validation"],
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )

In [ ]:
# trainer.train()
# from accelerate import Accelerator
# accelerator = Accelerator()
# trainer = accelerator.prepare(trainer)
trainer.train()


In [ ]:
trainer.save_model("outputs/models")

In [ ]:
stop

In [ ]:
# Load the trained model from saved checkpoint
model = AutoModelForCausalLM.from_pretrained("outputs/models")
model = model.to("cuda")

In [ ]:
# from pathlib import Path
# import json

# metrics = trainer.evaluate(ds["test"].select(range(100)))
# Path(training_args.output_dir).mkdir(parents=True, exist_ok=True)
# (Path(training_args.output_dir) / "test_metrics.json").write_text(json.dumps(metrics, indent=2))
# print("Test metrics saved to", training_args.output_dir)


In [ ]:
for k, v in metrics.items():
    print(f"{k}: {v:.4f}")


In [ ]:
# first_block = model.base_model.model.transformer.h[0]         # First transformer block
# attn = first_block.attn                                       # Attention module
# c_attn = attn.c_attn                                          # The fused QKV projection
# print(f"c_attn.weight shape: {c_attn.weight.shape}")          # Should be [3072, 1024]

In [ ]:
import torch
import time


In [ ]:
a = torch.randn(1024)
# b = torch.randn(1024)
b = torch.ones(1024)
rand_matrix = torch.randn(1024, 1024)

In [ ]:
a = torch.tensor([1,2,3,4,5])
b = torch.tensor([10,11,12])

In [ ]:
c = torch.tensor([[1,2,3],[4,5,6]])

In [ ]:
c

In [ ]:
c @ torch.diag(torch.tensor([1,2,3]))

In [ ]:
ab = torch.outer(a,b)

In [ ]:
print(ab)

In [ ]:
torch.allclose(torch.mul(rand_matrix, ab), (a[:, None] * rand_matrix) * b[None, :], atol=1e-5, rtol=1e-5)

In [ ]:
torch.allclose(rand_matrix * b.unsqueeze(1) * a.unsqueeze(0), b.unsqueeze(1) * rand_matrix * a.unsqueeze(0), atol=1e-5, rtol=1e-5)

In [ ]:
torch.allclose(rand_matrix + rand_matrix * a.unsqueeze(1) * b.unsqueeze(0),
                torch.mul(1+ab, rand_matrix), atol=1e-5, rtol=1e-5)

In [ ]:
def naive_calc(a, b, rand_matrix):
    return b.unsqueeze(1) * rand_matrix * a.unsqueeze(0)

In [ ]:
def vectorized(a, b, rand_matrix):
    ab = torch.outer(a,b)
    return torch.mul(rand_matrix, ab)

In [ ]:
def calc_major1(a, b, rand_matrix):
    return rand_matrix + rand_matrix * a.unsqueeze(1) * b.unsqueeze(0)

In [ ]:
def calc_major2(a, b, rand_matrix):
    ab = torch.outer(a,b)
    return torch.mul(1+ab, rand_matrix)

In [ ]:
times = []
for _ in range(1000):
    torch.cuda.synchronize()
    start = time.time()
    calc_major1(a, b, rand_matrix)
    torch.cuda.synchronize()
    end = time.time()
    times.append(end - start)

print(f"calc_major1 Avg: {sum(times)/len(times):.6f} sec | Std: {torch.std(torch.tensor(times)).item():.6f}")

times = []
for _ in range(1000):
    torch.cuda.synchronize()
    start = time.time()
    calc_major2(a, b, rand_matrix)
    torch.cuda.synchronize()
    end = time.time()
    times.append(end - start)

print(f"calc_major2 Avg: {sum(times)/len(times):.6f} sec | Std: {torch.std(torch.tensor(times)).item():.6f}")

In [ ]:
times = []
for _ in range(10000):
    torch.cuda.synchronize()
    start = time.time()
    naive_calc(a, b, rand_matrix)
    torch.cuda.synchronize()
    end = time.time()
    times.append(end - start)

print(f"naive_calc Avg: {sum(times)/len(times):.6f} sec | Std: {torch.std(torch.tensor(times)).item():.6f}")

In [ ]:
times = []
for _ in range(1000):
    torch.cuda.synchronize()
    start = time.time()
    vectorized(a, b, rand_matrix)
    torch.cuda.synchronize()
    end = time.time()
    times.append(end - start)

print(f"vectorized Avg: {sum(times)/len(times):.6f} sec | Std: {torch.std(torch.tensor(times)).item():.6f}")

In [ ]:
torch.norm